# A simple Colab example to help you quickly try out TowerMind!

## 1. Prepare the TowerMind repository.


In [ ]:
# Download the repository (can take few mins)
!git clone https://github.com/tb6147877/TowerMind.git

# Decompressing the linux based running environment
!unzip ./TowerMind/compressed_env/linux.zip -d ./TowerMind/compressed_env

# Grant execution permission.
!chmod +x ./TowerMind/compressed_env/linux/td.x86_64

## 2. Setup the Conda Environment.


In [ ]:
%%capture

# Download Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh

# Install Miniconda
!bash miniconda.sh -b -f -p /opt/conda

# Accept Anaconda Terms of Service
!/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Create TowerMind environment with Python 3.10.12
!/opt/conda/bin/conda create -n towermind python=3.10.12 -y

## 3. Install related libraries.

In [ ]:
!apt-get update -qq
!apt-get install -y xvfb
!/opt/conda/bin/conda run -n towermind pip install imageio imageio-ffmpeg pillow numpy
!/opt/conda/bin/conda run -n towermind pip install mlagents==1.1.0

## 4. Run code (Random Policy).

In [ ]:
code = '''
import imageio
import numpy as np
from PIL import Image
from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

unity_env = UnityEnvironment("/content/TowerMind/compressed_env/linux/td.x86_64")

env = UnityToGymWrapper(unity_env, uint8_visual=True)

with imageio.get_writer("example.mp4", fps=30) as video:
  done = False
  state = env.reset()
  frame = np.transpose(env.render().squeeze(), (1, 2, 0))
  video.append_data(frame)

  while not done:
    action = env.action_space.sample()
    state, _, done, _ = env.step(action)
    frame = np.transpose(env.render().squeeze(), (1, 2, 0))
    video.append_data(frame)

env.close()
'''

with open("run_in_conda.py", "w") as f:
    f.write(code)

!xvfb-run -s "-screen 0 1400x900x24" /opt/conda/envs/towermind/bin/python run_in_conda.py

## 5.Play this video!

In [ ]:
from IPython.display import Video
Video("example.mp4", embed=True)